# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import ast
import json
from typing import List

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [3]:
load_dotenv(find_dotenv(), override=True)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [4]:
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)

chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay", embedding_function=embedding_fn)

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

print(f"Loaded collection 'udaplay' with {collection.count()} documents")

Loaded collection 'udaplay' with 15 documents


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...)
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(query_texts=[query], n_results=3)
    docs = []
    for doc_id, meta, dist in zip(
        results["ids"][0], results["metadatas"][0], results["distances"][0]
    ):
        docs.append({
            "id": doc_id,
            "Platform": meta.get("Platform"),
            "Name": meta.get("Name"),
            "YearOfRelease": meta.get("YearOfRelease"),
            "Description": meta.get("Description"),
            "Genre": meta.get("Genre"),
            "Publisher": meta.get("Publisher"),
            "distance": round(dist, 4),
        })
    return docs

#### Evaluate Retrieval Tool

In [6]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Detailed explanation about the evaluation result")


@tool
def evaluate_retrieval(question: str, retrieved_docs: List[str]) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    llm = LLM(model="gpt-4o-mini", temperature=0)
    prompt = (
        "You are an expert evaluator of a video game knowledge base.\n"
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n"
        "Only mark the documents as useful if they contain the specific facts needed "
        "to answer the question fully; do not rely on your own knowledge.\n\n"
        f"Question: {question}\n\n"
        f"Retrieved documents:\n{json.dumps(retrieved_docs, indent=2)}"
    )
    response = llm.invoke(prompt, response_format=EvaluationReport)
    report = EvaluationReport.model_validate_json(response.content)
    return report.model_dump()

#### Game Web Search Tool

In [7]:
@tool
def game_web_search(question: str) -> dict:
    """
    Web search: Searches the web for information about games and the game industry.
    Use it only when the internal knowledge base is not enough to answer.
    args:
    - question: a question about game industry.
    """
    response = tavily_client.search(
        query=question,
        search_depth="advanced",
        max_results=5,
        include_answer=True,
        exclude_domains=["facebook.com", "reddit.com", "x.com", "twitter.com", "youtube.com"],
    )
    return {
        "answer": response.get("answer"),
        "results": [
            {
                "title": r.get("title"),
                "url": r.get("url"),
                "content": r.get("content"),
            }
            for r in response.get("results", [])
        ],
    }

### Agent

In [8]:
instructions = """
You are UdaPlay, an AI research agent specialized in the video game industry.

For every user question about games, follow this workflow strictly:
1. Call `retrieve_game` with the question to search the internal knowledge base.
2. Call `evaluate_retrieval` with the original question and ALL the retrieved documents
   (pass each document as a JSON string).
3. If the evaluation says the documents are NOT useful, call `game_web_search`.
   Never call `game_web_search` if the internal documents were judged useful.
4. Write the final answer.

If a question refers to something from earlier in the conversation (e.g. "that game"),
resolve the reference using the conversation history before searching.

Format the final answer exactly like this:

**Answer:** <clear, natural-language answer>

**Reasoning:** <one or two sentences on which tools were used and why>

**Confidence:** <High | Medium | Low>

**Sources:**
- <for internal knowledge: "Internal knowledge base - <game name> (doc id <id>)">
- <for web results: "<page title> - <URL>">

Copy source URLs exactly as returned by `game_web_search`.
Only use facts from the tool results. If you cannot find the answer, say so.
"""

udaplay_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.0,
)

In [9]:
def run_query(agent, query, session_id="udaplay_session"):
    print("=" * 100)
    print(f"USER: {query}")
    print("=" * 100)
    run = agent.invoke(query, session_id=session_id)
    messages = run.get_final_state()["messages"]

    # Only show the messages produced for this query
    last_user_idx = max(i for i, m in enumerate(messages) if isinstance(m, UserMessage))
    for m in messages[last_user_idx + 1:]:
        if isinstance(m, AIMessage) and m.tool_calls:
            for call in m.tool_calls:
                print(f"\n[TOOL CALL] {call.function.name}({call.function.arguments})")
        elif isinstance(m, ToolMessage) and m.name == "game_web_search":
            result = ast.literal_eval(json.loads(m.content))
            print(f"[TOOL RESULT] game_web_search answer: {result['answer']}")
            for r in result["results"]:
                print(f"    - {r['title']} | {r['url']}")
        elif isinstance(m, ToolMessage):
            content = json.loads(m.content)
            if len(content) > 1500:
                content = content[:1500] + " ..."
            print(f"[TOOL RESULT] {m.name}: {content}")
        elif isinstance(m, AIMessage):
            print(f"\n[FINAL ANSWER]\n{m.content}\n")
    return run


queries = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X realeased for Playstation 5?",
]

for q in queries:
    run_query(udaplay_agent, q)

USER: When was Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

[TOOL CALL] retrieve_game({"query":"Pokémon Gold and Silver release date"})
[TOOL RESULT] retrieve_game: [{'id': '006', 'Platform': 'Game Boy Color', 'Name': 'Pokémon Gold and Silver', 'YearOfRelease': 1999, 'Description': 'Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.', 'Genre': 'Role-playing', 'Publisher': 'Nintendo', 'distance': 0.1268}, {'id': '007', 'Platform': 'Game Boy Advance', 'Name': 'Pokémon Ruby and Sapphire', 'YearOfRelease': 2002, 'Description': 'Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.', 'Genre': 'Role-playing', 'Publisher': 'Nintendo', 'distance': 0.161}, {'id': '012', 'Platform': 'Nintendo Switch', 'Name': 'Mario Kart 8 Deluxe', 'YearOfRelease': 2017, 'Description': 'An enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics.', 'Genre': '

... (8 outputs left out)

[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

[TOOL CALL] retrieve_game({"query":"Mortal Kombat X release for Playstation 5"})
[TOOL RESULT] retrieve_game: [{'id': '005', 'Platform': 'PlayStation 5', 'Name': "Marvel's Spider-Man 2", 'YearOfRelease': 2023, 'Description': 'The sequel to the acclaimed Spider-Man game, featuring both Peter Parker and Miles Morales as playable characters.', 'Genre': 'Action-adventure', 'Publisher': 'Sony Interactive Entertainment', 'distance': 0.1913}, {'id': '015', 'Platform': 'Xbox Series X|S', 'Name': 'Halo Infinite', 'YearOfRelease': 2021, 'Description': "The latest installment in the Halo franchise, featuring Master Chief's return in a new open-world setting.", 'Genre': 'First-person shooter', 'Publisher': 'Xbox Game Studios', 'distance': 0.2072}, {'id': '004', 'Platform': 'PlayStation 4', 'Name': "Marvel's Spider-Man", 'YearOfRelease': 2018, 'Description': 'An open-world superhero game that lets players swin

In [10]:
# Follow-up question in the same session: the agent uses the conversation state
run_query(udaplay_agent, "What platform was that Mario game launched on, and who published it?")

USER: What platform was that Mario game launched on, and who published it?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

[TOOL CALL] retrieve_game({"query":"Super Mario 64 platform and publisher"})
[TOOL RESULT] retrieve_game: [{'id': '009', 'Platform': 'Nintendo 64', 'Name': 'Super Mario 64', 'YearOfRelease': 1996, 'Description': "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.", 'Genre': 'Platformer', 'Publisher': 'Nintendo', 'distance': 0.0892}, {'id': '008', 'Platform': 'Super Nintendo Entertainment System (SNES)', 'Name': 'Super Mario World', 'YearOfRelease': 1990, 'Description': 'A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.', 'Genre': 'Platformer', 'Publisher': 'Nintendo', 'distance': 0.1234}, {'id': '012', 'Platform': 'Nintendo Switch', 'Name': 'Mario Kart 8 Deluxe', 'YearOfRelease': 2017, 'Description': 'An enhanced version of Mario Kart 8, featuring new characters, tracks, 

Run('2f5ab67a-9a2b-4201-8fab-e6ee1dfee51f')

### (Optional) Advanced

In [11]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes